# 02. Data Preprocessing – German Credit Dataset (Google Colab)

## 1. Objective

This notebook prepares the German Credit dataset for baseline modelling and later explainability analysis.

The preprocessing stage converts the raw analytical dataset into clean, reproducible, model-ready train and test datasets. It also saves the original train/test splits so that later explainability outputs can be traced back to the untransformed input records.

## 1.1 Dissertation Alignment

This notebook forms part of a Master's dissertation research pipeline on responsible machine learning and explainable AI in high-stakes decision systems.

The original proposal focused on interpretable and ethical machine learning. This implementation strengthens the proposal by ensuring that preprocessing is:

- reproducible,
- leakage-aware,
- traceable from raw records to model-ready features,
- reusable across German Credit, recidivism, and diabetes datasets,
- aligned to later SHAP, LIME, DiCE, and EES Framework notebooks.

This is appropriate for a Master's dissertation because the contribution is methodological, empirical, and defensible without overstating the work as a PhD-level theoretical contribution.

## 2. Key Assumptions

The following assumptions are applied in this notebook:

1. The Kaggle feature file and the UCI target file have the same row order.
2. The original German Credit target is coded as:
   - `1 = Good credit`
   - `2 = Bad credit`
3. For this research, the target is transformed into a risk/modelling frame:
   - `0 = Good credit / favourable outcome`
   - `1 = Bad credit / unfavourable outcome`
4. The variable `Job` is treated as categorical, even though it is numerically coded.
5. Numeric missing values are imputed using the median.
6. Categorical missing values are retained as an explicit `Missing` category.
7. Numeric variables are scaled using `RobustScaler` because outliers are retained.
8. Categorical variables are encoded using one-hot encoding.
9. The preprocessing pipeline is fitted on the training set only to prevent data leakage.
10. Outputs are saved for downstream modelling, explainability, and EES Framework scoring.

## 3. Import Required Libraries

In [ ]:
# =========================================================
# 3. Import Required Libraries
# =========================================================

import os
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

import joblib

warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

# Plot settings
plt.rcParams["figure.figsize"] = (8, 4)

print("Libraries imported successfully.")

Libraries imported successfully.


## 4. Mount Google Drive and Define Project Paths

The same path structure should be reused across all German Credit notebooks.  
For recidivism and diabetes, only `DATASET_NAME` and `NOTEBOOK_NAME` should change.

In [ ]:
# =========================================================
# 4. Mount Google Drive
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================================
# 4.1 Project Path Configuration
# =========================================================

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = "german_credit"
NOTEBOOK_NAME = "02_data_preprocessing_german_credit"

# Notebook code location
CODE_PATH = ROOT_PATH / "05_Code" / "Github_Repo" / "notebooks" / DATASET_NAME

# Notebook-specific results folder
RESULTS_PATH = ROOT_PATH / "07_Results" / DATASET_NAME / NOTEBOOK_NAME

# Standard output folders for this notebook
TABLES_PATH = RESULTS_PATH / "tables"
FIGURES_PATH = RESULTS_PATH / "figures"
DATASETS_PATH = RESULTS_PATH / "datasets"
OBJECTS_PATH = RESULTS_PATH / "models"
LOGS_PATH = RESULTS_PATH / "logs"

# Shared dataset folder used by downstream notebooks
SHARED_DATASETS_PATH = ROOT_PATH / "07_Results" / "Datasets" / DATASET_NAME

# Create output folders
for path in [
    RESULTS_PATH,
    TABLES_PATH,
    FIGURES_PATH,
    DATASETS_PATH,
    OBJECTS_PATH,
    LOGS_PATH,
    SHARED_DATASETS_PATH
]:
    path.mkdir(parents=True, exist_ok=True)

print("Project paths configured successfully.")
print(f"ROOT_PATH: {ROOT_PATH}")
print(f"RESULTS_PATH: {RESULTS_PATH}")
print(f"SHARED_DATASETS_PATH: {SHARED_DATASETS_PATH}")

Project paths configured successfully.
ROOT_PATH: /content/drive/MyDrive/Master_Research
RESULTS_PATH: /content/drive/MyDrive/Master_Research/07_Results/german_credit/02_data_preprocessing_german_credit
SHARED_DATASETS_PATH: /content/drive/MyDrive/Master_Research/07_Results/Datasets/german_credit


## 5. Define Raw Data Paths

The notebook first checks the preferred raw data location.  
A fallback path is included to support previous folder versions without breaking the notebook.

In [ ]:
# =========================================================
# 5. Raw Data Path Configuration
# =========================================================

candidate_data_paths = [
    ROOT_PATH / "03_Data" / "01_Raw_Data" / DATASET_NAME / "german_credit_data.csv",
    ROOT_PATH / "04_Data" / "01_Raw_Data" / DATASET_NAME / "german_credit_data.csv",
    ROOT_PATH / "03_Data" / "01_Raw_Data" / "german_credit_data.csv",
    ROOT_PATH / "04_Data" / "01_Raw_Data" / "german_credit_data.csv",
]

DATA_PATH = None

for candidate_path in candidate_data_paths:
    if candidate_path.exists():
        DATA_PATH = candidate_path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "The German Credit feature file was not found. "
        "Please place german_credit_data.csv in one of the expected raw data folders."
    )

print(f"Feature dataset path found: {DATA_PATH}")

Feature dataset path found: /content/drive/MyDrive/Master_Research/04_Data/01_Raw_Data/german_credit/german_credit_data.csv


## 6. Load Feature Dataset

The Kaggle feature dataset is loaded first.  
Any unwanted index column such as `Unnamed: 0` is removed.

In [ ]:
# =========================================================
# 6. Load Features Dataset
# =========================================================

df_features = pd.read_csv(DATA_PATH)

# Drop accidental index columns if present
unnamed_cols = [col for col in df_features.columns if col.lower().startswith("unnamed")]
if unnamed_cols:
    df_features = df_features.drop(columns=unnamed_cols)

print("Feature dataset loaded successfully.")
print(f"Rows: {df_features.shape[0]}")
print(f"Columns: {df_features.shape[1]}")

display(df_features.head())

Feature dataset loaded successfully.
Rows: 1000
Columns: 9


,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,67,male,2,own,NaN,little,1169,6,radio/TV
1,22,female,2,own,little,moderate,5951,48,radio/TV
2,49,male,1,own,little,NaN,2096,12,education
3,45,male,2,free,little,little,7882,42,furniture/equipment
4,53,male,2,free,little,little,4870,24,car


## 7. Load Target Variable from UCI Repository

The target variable is loaded from the UCI Machine Learning Repository using `ucimlrepo`.  
This supports reproducibility and avoids manually creating the target variable.

In [ ]:
# =========================================================
# 7. Install and Load UCI Target Data
# =========================================================

try:
    from ucimlrepo import fetch_ucirepo
except ImportError:
    !pip install -q ucimlrepo
    from ucimlrepo import fetch_ucirepo

statlog_german_credit_data = fetch_ucirepo(id=144)

df_target = statlog_german_credit_data.data.targets.copy()
df_target = df_target.rename(columns={df_target.columns[0]: "class"})

print("Target dataset loaded successfully.")
print(f"Rows: {df_target.shape[0]}")
print(f"Columns: {df_target.shape[1]}")

display(df_target.head())

Target dataset loaded successfully.
Rows: 1000
Columns: 1


,class
0,1
1,2
2,1
3,1
4,2


## 8. Validate and Combine Features with Target

This validation step checks that the feature dataset and target dataset have the same number of rows before combining them.

In [ ]:
# =========================================================
# 8. Validate and Combine Dataset
# =========================================================

if len(df_features) != len(df_target):
    raise ValueError(
        f"Row mismatch detected: features have {len(df_features)} rows, "
        f"but target has {len(df_target)} rows."
    )

df = df_features.copy()
df["class"] = df_target["class"].values

print("Combined analytical dataset created successfully.")
print(f"Shape: {df.shape}")

display(df.head())

Combined analytical dataset created successfully.
Shape: (1000, 10)


,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,class
0,67,male,2,own,NaN,little,1169,6,radio/TV,1
1,22,female,2,own,little,moderate,5951,48,radio/TV,2
2,49,male,1,own,little,NaN,2096,12,education,1
3,45,male,2,free,little,little,7882,42,furniture/equipment,1
4,53,male,2,free,little,little,4870,24,car,2


## 9. Prepare Target Variable

The target is recoded to support risk-based binary classification:

- `0` = good credit / favourable outcome
- `1` = bad credit / unfavourable outcome

This framing is important because later explainability, counterfactual, and EES calculations focus on unfavourable/high-risk predictions.

In [ ]:
# =========================================================
# 9. Prepare Target Variable
# =========================================================

original_target_values = sorted(df["class"].dropna().unique().tolist())
print("Original target values:", original_target_values)

if original_target_values == [1, 2]:
    df["target"] = df["class"].map({1: 0, 2: 1})
else:
    # This allows the notebook to keep working if the target is already binary.
    df["target"] = df["class"]

df = df.drop(columns=["class"])

target_distribution = (
    df["target"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="count")
)

target_distribution["percentage"] = (
    target_distribution["count"] / target_distribution["count"].sum() * 100
).round(2)

print("Target distribution after recoding:")
display(target_distribution)

Original target values: [1, 2]
Target distribution after recoding:


,target,count,percentage
0,0,700,70.0
1,1,300,30.0


## 10. Separate Features and Target

In [ ]:
# =========================================================
# 10. Separate Features and Target
# =========================================================

X = df.drop(columns=["target"]).copy()
y = df["target"].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

Feature matrix shape: (1000, 9)
Target vector shape: (1000,)


## 11. Identify Feature Types

Features are separated into numeric and categorical groups so that appropriate preprocessing can be applied.

In [ ]:
# =========================================================
# 11. Identify Feature Types
# =========================================================

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Initial numeric columns:")
print(numeric_cols)

print("\nInitial categorical columns:")
print(categorical_cols)

Initial numeric columns:
['Age', 'Job', 'Credit amount', 'Duration']

Initial categorical columns:
['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']


## 12. Treat `Job` as a Categorical Feature

Although `Job` appears numeric, it represents occupational categories rather than a continuous measurement.  
Therefore, it is moved from the numeric pipeline to the categorical pipeline.

In [ ]:
# =========================================================
# 12. Adjust Job Feature Treatment
# =========================================================

if "Job" in numeric_cols:
    numeric_cols.remove("Job")

if "Job" in X.columns and "Job" not in categorical_cols:
    categorical_cols.append("Job")

print("Final numeric columns:")
print(numeric_cols)

print("\nFinal categorical columns:")
print(categorical_cols)

Final numeric columns:
['Age', 'Credit amount', 'Duration']

Final categorical columns:
['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose', 'Job']


## 13. Create Feature Audit Table for Preprocessing

This table documents each feature's preprocessing treatment.  
It supports methodological transparency and can be reused in the dissertation methodology chapter.

In [ ]:
# =========================================================
# 13. Feature Audit Table
# =========================================================

feature_audit = []

for col in X.columns:
    if col in numeric_cols:
        feature_type = "numeric"
        imputation = "median"
        scaling = "RobustScaler"
        encoding = "not applied"
    elif col in categorical_cols:
        feature_type = "categorical"
        imputation = "constant: Missing"
        scaling = "not applied"
        encoding = "OneHotEncoder(handle_unknown='ignore')"
    else:
        feature_type = "not assigned"
        imputation = "not assigned"
        scaling = "not assigned"
        encoding = "not assigned"

    feature_audit.append({
        "feature": col,
        "assigned_feature_type": feature_type,
        "original_dtype": str(X[col].dtype),
        "missing_count": int(X[col].isna().sum()),
        "missing_percentage": round(float(X[col].isna().mean() * 100), 2),
        "unique_values": int(X[col].nunique(dropna=True)),
        "imputation_method": imputation,
        "scaling_applied": scaling,
        "encoding_applied": encoding,
        "notes": "Job treated as categorical" if col == "Job" else ""
    })

feature_audit_df = pd.DataFrame(feature_audit)

feature_audit_path = TABLES_PATH / "preprocessing_feature_audit.xlsx"
feature_audit_df.to_excel(feature_audit_path, index=False)

print(f"Feature audit saved to: {feature_audit_path}")
display(feature_audit_df)

Feature audit saved to: /content/drive/MyDrive/Master_Research/07_Results/german_credit/02_data_preprocessing_german_credit/tables/preprocessing_feature_audit.xlsx


,feature,assigned_feature_type,original_dtype,missing_count,missing_percentage,unique_values,imputation_method,scaling_applied,encoding_applied,notes
0,Age,numeric,int64,0,0.0,53,median,RobustScaler,not applied,
1,Sex,categorical,object,0,0.0,2,constant: Missing,not applied,OneHotEncoder(handle_unknown='ignore'),
2,Job,categorical,int64,0,0.0,4,constant: Missing,not applied,OneHotEncoder(handle_unknown='ignore'),Job treated as categorical
3,Housing,categorical,object,0,0.0,3,constant: Missing,not applied,OneHotEncoder(handle_unknown='ignore'),
4,Saving accounts,categorical,object,183,18.3,4,constant: Missing,not applied,OneHotEncoder(handle_unknown='ignore'),
5,Checking account,categorical,object,394,39.4,3,constant: Missing,not applied,OneHotEncoder(handle_unknown='ignore'),
6,Credit amount,numeric,int64,0,0.0,921,median,RobustScaler,not applied,
7,Duration,numeric,int64,0,0.0,33,median,RobustScaler,not applied,
8,Purpose,categorical,object,0,0.0,8,constant: Missing,not applied,OneHotEncoder(handle_unknown='ignore'),


## 14. Define Preprocessing Pipelines

### Numeric Pipeline

- Median imputation
- Robust scaling

### Categorical Pipeline

- Missing values replaced with `Missing`
- One-hot encoding with unknown-category protection

In [ ]:
# =========================================================
# 14. Define Preprocessing Pipelines
# =========================================================

# Compatibility helper for different scikit-learn versions
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("encoder", one_hot_encoder)
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ],
    remainder="drop"
)

print("Preprocessing pipelines defined successfully.")

Preprocessing pipelines defined successfully.


## 15. Train/Test Split

The dataset is split into:

- 80% training data
- 20% testing data

Stratified sampling is used to preserve the target distribution in both sets.

In [ ]:
# =========================================================
# 15. Train/Test Split
# =========================================================

TEST_SIZE = 0.20
RANDOM_STATE = 42

X_train_original, X_test_original, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train/test split completed.")
print(f"X_train_original shape: {X_train_original.shape}")
print(f"X_test_original shape: {X_test_original.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Train/test split completed.
X_train_original shape: (800, 9)
X_test_original shape: (200, 9)
y_train shape: (800,)
y_test shape: (200,)


## 16. Validate Target Distribution After Split

In [ ]:
# =========================================================
# 16. Validate Target Distribution After Split
# =========================================================

split_distribution = pd.DataFrame({
    "full_dataset": y.value_counts(normalize=True).sort_index(),
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index()
}).reset_index().rename(columns={"index": "target"})

for col in ["full_dataset", "train", "test"]:
    split_distribution[col] = (split_distribution[col] * 100).round(2)

split_distribution_path = TABLES_PATH / "target_distribution_after_split.xlsx"
split_distribution.to_excel(split_distribution_path, index=False)

print(f"Target distribution saved to: {split_distribution_path}")
display(split_distribution)

Target distribution saved to: /content/drive/MyDrive/Master_Research/07_Results/german_credit/02_data_preprocessing_german_credit/tables/target_distribution_after_split.xlsx


,target,full_dataset,train,test
0,0,70.0,70.0,70.0
1,1,30.0,30.0,30.0


## 17. Fit Preprocessor on Training Data Only

The preprocessor is fitted only on the training data to prevent data leakage.

In [ ]:
# =========================================================
# 17. Fit and Transform Data
# =========================================================

X_train_processed = preprocessor.fit_transform(X_train_original)
X_test_processed = preprocessor.transform(X_test_original)

print("Preprocessing applied successfully.")
print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed testing shape: {X_test_processed.shape}")

Preprocessing applied successfully.
Processed training shape: (800, 29)
Processed testing shape: (200, 29)


## 18. Retrieve Transformed Feature Names

The transformed feature names are needed for model interpretation and explainability.

In [ ]:
# =========================================================
# 18. Retrieve Transformed Feature Names
# =========================================================

cat_feature_names = (
    preprocessor
    .named_transformers_["cat"]
    .named_steps["encoder"]
    .get_feature_names_out(categorical_cols)
)

feature_names = numeric_cols + list(cat_feature_names)

X_train_preprocessed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train_original.index
)

X_test_preprocessed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test_original.index
)

print(f"Number of transformed features: {len(feature_names)}")
display(X_train_preprocessed.head())

Number of transformed features: 29


,Age,Credit amount,Duration,Sex_female,Sex_male,Housing_free,Housing_own,Housing_rent,Saving accounts_Missing,Saving accounts_little,Saving accounts_moderate,Saving accounts_quite rich,Saving accounts_rich,Checking account_Missing,Checking account_little,Checking account_moderate,Checking account_rich,Purpose_business,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others,Job_0,Job_1,Job_2,Job_3
828,1.000000,2.332558,1.5,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
997,0.357143,-0.586434,-0.5,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
148,-0.357143,1.183721,1.5,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
735,-0.285714,0.648450,1.5,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
130,-0.642857,2.391473,2.5,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## 19. Validate Preprocessed Outputs

This validation confirms that the transformed datasets are model-ready.

In [ ]:
# =========================================================
# 19. Validate Preprocessed Outputs
# =========================================================

validation_summary = pd.DataFrame([
    {
        "dataset": "X_train_preprocessed",
        "rows": X_train_preprocessed.shape[0],
        "columns": X_train_preprocessed.shape[1],
        "missing_values": int(X_train_preprocessed.isna().sum().sum()),
        "duplicate_rows": int(X_train_preprocessed.duplicated().sum())
    },
    {
        "dataset": "X_test_preprocessed",
        "rows": X_test_preprocessed.shape[0],
        "columns": X_test_preprocessed.shape[1],
        "missing_values": int(X_test_preprocessed.isna().sum().sum()),
        "duplicate_rows": int(X_test_preprocessed.duplicated().sum())
    }
])

validation_path = TABLES_PATH / "preprocessing_validation_summary.xlsx"
validation_summary.to_excel(validation_path, index=False)

print(f"Validation summary saved to: {validation_path}")
display(validation_summary)

if validation_summary["missing_values"].sum() > 0:
    raise ValueError("Missing values remain after preprocessing.")

Validation summary saved to: /content/drive/MyDrive/Master_Research/07_Results/german_credit/02_data_preprocessing_german_credit/tables/preprocessing_validation_summary.xlsx


,dataset,rows,columns,missing_values,duplicate_rows
0,X_train_preprocessed,800,29,0,0
1,X_test_preprocessed,200,29,0,0


## 20. Create Split Index Mapping

The original row index is preserved to support traceability between:

- original records,
- preprocessed records,
- model predictions,
- SHAP explanations,
- LIME explanations,
- DiCE counterfactuals,
- EES scoring outputs.

In [ ]:
# =========================================================
# 20. Create Split Index Mapping
# =========================================================

train_mapping = pd.DataFrame({
    "split": "train",
    "original_index": X_train_original.index,
    "target": y_train.values
})

test_mapping = pd.DataFrame({
    "split": "test",
    "original_index": X_test_original.index,
    "target": y_test.values
})

split_index_mapping = pd.concat([train_mapping, test_mapping], ignore_index=True)

split_mapping_path = TABLES_PATH / "split_index_mapping.xlsx"
split_index_mapping.to_excel(split_mapping_path, index=False)

print(f"Split index mapping saved to: {split_mapping_path}")
display(split_index_mapping.head())

Split index mapping saved to: /content/drive/MyDrive/Master_Research/07_Results/german_credit/02_data_preprocessing_german_credit/tables/split_index_mapping.xlsx


,split,original_index,target
0,train,828,1
1,train,997,0
2,train,148,0
3,train,735,0
4,train,130,0


## 21. Save Preprocessed and Original Datasets

The notebook saves outputs in two locations:

1. Notebook-specific result folder  
2. Shared dataset folder for downstream notebooks

This allows each notebook to remain self-contained while also supporting the full modelling and EES pipeline.

In [ ]:
# =========================================================
# 21. Save Datasets
# =========================================================

def save_dataset_outputs(output_path: Path):
    output_path.mkdir(parents=True, exist_ok=True)

    # Preprocessed model-ready data
    X_train_preprocessed.to_csv(output_path / "X_train_preprocessed.csv", index=True, index_label="original_index")
    X_test_preprocessed.to_csv(output_path / "X_test_preprocessed.csv", index=True, index_label="original_index")

    # Original split data for interpretation and traceability
    X_train_original.to_csv(output_path / "X_train_original.csv", index=True, index_label="original_index")
    X_test_original.to_csv(output_path / "X_test_original.csv", index=True, index_label="original_index")

    # Target splits
    y_train.to_frame("target").to_csv(output_path / "y_train.csv", index=True, index_label="original_index")
    y_test.to_frame("target").to_csv(output_path / "y_test.csv", index=True, index_label="original_index")

    # Feature names
    pd.DataFrame({"feature_name": feature_names}).to_csv(output_path / "preprocessed_feature_names.csv", index=False)

    # Split mapping
    split_index_mapping.to_csv(output_path / "split_index_mapping.csv", index=False)

save_dataset_outputs(DATASETS_PATH)
save_dataset_outputs(SHARED_DATASETS_PATH)

print("Dataset outputs saved successfully.")
print(f"Notebook datasets path: {DATASETS_PATH}")
print(f"Shared datasets path: {SHARED_DATASETS_PATH}")

Dataset outputs saved successfully.
Notebook datasets path: /content/drive/MyDrive/Master_Research/07_Results/german_credit/02_data_preprocessing_german_credit/datasets
Shared datasets path: /content/drive/MyDrive/Master_Research/07_Results/Datasets/german_credit


## 22. Save Preprocessing Object

The fitted preprocessing object is saved so that the same transformation can be reused consistently in modelling, explainability, and deployment-style workflows.

In [ ]:
# =========================================================
# 22. Save Preprocessor Object
# =========================================================

preprocessor_path = OBJECTS_PATH / "german_credit_preprocessor.pkl"
shared_preprocessor_path = SHARED_DATASETS_PATH / "german_credit_preprocessor.pkl"

joblib.dump(preprocessor, preprocessor_path)
joblib.dump(preprocessor, shared_preprocessor_path)

print(f"Preprocessor saved to: {preprocessor_path}")
print(f"Shared preprocessor saved to: {shared_preprocessor_path}")

Preprocessor saved to: /content/drive/MyDrive/Master_Research/07_Results/german_credit/02_data_preprocessing_german_credit/objects/german_credit_preprocessor.pkl
Shared preprocessor saved to: /content/drive/MyDrive/Master_Research/07_Results/Datasets/german_credit/german_credit_preprocessor.pkl


## 23. Save Preprocessing Metadata

Metadata is saved to make the preprocessing assumptions explicit and reproducible.

In [ ]:
# =========================================================
# 23. Save Metadata
# =========================================================

metadata = {
    "dataset_name": DATASET_NAME,
    "notebook_name": NOTEBOOK_NAME,
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "root_path": str(ROOT_PATH),
    "raw_data_path": str(DATA_PATH),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "target_mapping": {
        "original_1": "0 = Good credit / favourable outcome",
        "original_2": "1 = Bad credit / unfavourable outcome"
    },
    "numeric_columns": numeric_cols,
    "categorical_columns": categorical_cols,
    "n_original_features": int(X.shape[1]),
    "n_preprocessed_features": int(len(feature_names)),
    "train_rows": int(X_train_preprocessed.shape[0]),
    "test_rows": int(X_test_preprocessed.shape[0]),
    "preprocessing_choices": {
        "numeric_imputation": "median",
        "numeric_scaling": "RobustScaler",
        "categorical_imputation": "constant: Missing",
        "categorical_encoding": "OneHotEncoder(handle_unknown='ignore')",
        "job_treatment": "categorical"
    }
}

metadata_path = LOGS_PATH / "preprocessing_metadata.json"
shared_metadata_path = SHARED_DATASETS_PATH / "preprocessing_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

with open(shared_metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Metadata saved to: {metadata_path}")
print(f"Shared metadata saved to: {shared_metadata_path}")

Metadata saved to: /content/drive/MyDrive/Master_Research/07_Results/german_credit/02_data_preprocessing_german_credit/logs/preprocessing_metadata.json
Shared metadata saved to: /content/drive/MyDrive/Master_Research/07_Results/Datasets/german_credit/preprocessing_metadata.json


## 24. Save Consolidated Excel Output

A consolidated Excel workbook is saved for documentation and review.

In [ ]:
# =========================================================
# 24. Save Consolidated Excel Workbook
# =========================================================

excel_output_path = TABLES_PATH / "data_preprocessing_summary_german_credit.xlsx"

with pd.ExcelWriter(excel_output_path, engine="openpyxl") as writer:
    target_distribution.to_excel(writer, sheet_name="target_distribution", index=False)
    split_distribution.to_excel(writer, sheet_name="split_distribution", index=False)
    feature_audit_df.to_excel(writer, sheet_name="feature_audit", index=False)
    validation_summary.to_excel(writer, sheet_name="validation_summary", index=False)
    split_index_mapping.to_excel(writer, sheet_name="split_index_mapping", index=False)
    pd.DataFrame({"feature_name": feature_names}).to_excel(writer, sheet_name="feature_names", index=False)

print(f"Consolidated preprocessing workbook saved to: {excel_output_path}")

Consolidated preprocessing workbook saved to: /content/drive/MyDrive/Master_Research/07_Results/german_credit/02_data_preprocessing_german_credit/tables/data_preprocessing_summary_german_credit.xlsx


## 25. Interpretation and Dissertation Notes

The preprocessing pipeline is now ready for baseline modelling.

Key methodological points:

- The target was recoded into a risk-based binary outcome, where `1` represents the unfavourable credit outcome.
- Missing values were handled inside the preprocessing pipeline, not manually outside the pipeline.
- The preprocessor was fitted on training data only, reducing leakage risk.
- Original splits were saved alongside transformed splits to preserve explainability traceability.
- The saved preprocessing object enables consistent reuse across modelling, SHAP, LIME, DiCE, and EES scoring.

These choices strengthen the dissertation by making the empirical workflow reproducible, auditable, and extendable to the recidivism and diabetes datasets.

## 26. Extension to Recidivism and Diabetes

To reuse this notebook for another dataset:

1. Change `DATASET_NAME`.
2. Change `NOTEBOOK_NAME`.
3. Update the raw data path.
4. Update the target preparation section.
5. Review which variables are numeric, categorical, sensitive, mutable, and immutable.
6. Keep the same output structure so downstream notebooks can run without major changes.

This supports the cross-domain benchmarking contribution of the dissertation.

## 27. Completion Check

In [ ]:
# =========================================================
# 27. Completion Check
# =========================================================

expected_files = [
    SHARED_DATASETS_PATH / "X_train_preprocessed.csv",
    SHARED_DATASETS_PATH / "X_test_preprocessed.csv",
    SHARED_DATASETS_PATH / "X_train_original.csv",
    SHARED_DATASETS_PATH / "X_test_original.csv",
    SHARED_DATASETS_PATH / "y_train.csv",
    SHARED_DATASETS_PATH / "y_test.csv",
    SHARED_DATASETS_PATH / "preprocessed_feature_names.csv",
    SHARED_DATASETS_PATH / "split_index_mapping.csv",
    SHARED_DATASETS_PATH / "german_credit_preprocessor.pkl",
    SHARED_DATASETS_PATH / "preprocessing_metadata.json"
]

completion_check = pd.DataFrame({
    "file": [str(file) for file in expected_files],
    "exists": [file.exists() for file in expected_files]
})

display(completion_check)

if completion_check["exists"].all():
    print("Preprocessing notebook completed successfully. All expected shared outputs were created.")
else:
    print("Some expected outputs are missing. Please review the completion check table.")

,file,exists
0,/content/drive/MyDrive/Master_Research/07_Resu...,True
1,/content/drive/MyDrive/Master_Research/07_Resu...,True
2,/content/drive/MyDrive/Master_Research/07_Resu...,True
3,/content/drive/MyDrive/Master_Research/07_Resu...,True
4,/content/drive/MyDrive/Master_Research/07_Resu...,True
5,/content/drive/MyDrive/Master_Research/07_Resu...,True
6,/content/drive/MyDrive/Master_Research/07_Resu...,True
7,/content/drive/MyDrive/Master_Research/07_Resu...,True
8,/content/drive/MyDrive/Master_Research/07_Resu...,True
9,/content/drive/MyDrive/Master_Research/07_Resu...,True


Preprocessing notebook completed successfully. All expected shared outputs were created.
